# 11. Spectrum Occupancy Trend Prediction using SSA and SVR

This notebook implements the **SSA-SVR** trend prediction method from the paper:
"A Trend Prediction Method for Failures Time Series Data by Exploring Singular Spectrum Analysis and Support Vector Machines Regression" (ICCSNT 2019).

**Pipeline:**
1. **SSA** (Singular Spectrum Analysis) — extract trend component from the raw series (reduces noise).
2. **SVR** (Support Vector Regression) — predict the trend using a sliding window; multi-step forecast by iteration.

**Same setup as notebook 10:** Same data (`work_dir/final`), same next-day prediction (last 72h → next 24h), same metrics (MAE, RMSE, MASE) and visual style.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from sklearn.svm import SVR
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt

print('Imports OK')

In [ ]:
# Use all CPU cores for SVR and data prep (sklearn uses n_jobs=-1)
import os
NUM_CORES = os.cpu_count() or 4
print(f'Using up to {NUM_CORES} CPU cores')

### Predict next day

- **Training:** 6 days. We use the **last 3 days (72 h)** as input and the **next 24 h (next day)** as target.
- **Testing:** 3 days — one sample per test day; input = previous 72 h, target = that day's 24 h.
- **Parameters:** `LOOKBACK = 72`, `FORECAST_HORIZON = 24`.

In [ ]:
work_dir = Path("work_dir")
if not work_dir.exists():
    work_dir = Path("../work_dir")
final_dir = work_dir / "final"
if not final_dir.exists():
    raise FileNotFoundError('Final dir not found: ' + str(final_dir))

training_dir = final_dir / "training"
testing_dir = final_dir / "testing"
if not training_dir.exists() or not testing_dir.exists():
    raise FileNotFoundError('Training or testing directory not found')

classes_found = [d.name for d in sorted(training_dir.iterdir()) if d.is_dir()]
class_options = sorted(set(classes_found))
print(f'Found {len(class_options)} classes (bands): {class_options}')

## Data loading and preparation

In [ ]:
def load_data_for_band(band_name: str, split: str = "training"):
    split_dir = final_dir / split / band_name
    if not split_dir.exists():
        return pd.DataFrame()
    dfs = []
    for p in sorted(split_dir.glob("final_*.parquet")):
        try:
            dfs.append(pd.read_parquet(p))
        except Exception as e:
            print(f'Error loading {p}: {e}')
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


def get_series_per_freq(train_df: pd.DataFrame, test_df: pd.DataFrame):
    """Return dict freq -> (train_series, test_series) for SSA-SVR."""
    thresholds = sorted(train_df["threshold_dbm"].unique())
    if len(thresholds) > 1:
        train_df = train_df[train_df["threshold_dbm"] == thresholds[0]].copy()
        test_df = test_df[test_df["threshold_dbm"] == thresholds[0]].copy()
    out = {}
    for freq in sorted(train_df["freq_center_ghz"].unique()):
        tr = train_df[train_df["freq_center_ghz"] == freq].sort_values(["date", "hour"])["au_pct"].values
        te = test_df[test_df["freq_center_ghz"] == freq].sort_values(["date", "hour"])["au_pct"].values
        if len(tr) > 0 and len(te) > 0:
            out[freq] = (tr, te)
    return out

In [ ]:
LOOKBACK = 72
FORECAST_HORIZON = 24

train_data_by_band = {}
test_data_by_band = {}
for band in class_options:
    train_df = load_data_for_band(band, "training")
    test_df = load_data_for_band(band, "testing")
    if not train_df.empty and not test_df.empty:
        train_data_by_band[band] = train_df
        test_data_by_band[band] = test_df
        print(f'Band {band}: Train={len(train_df)} rows, Test={len(test_df)} rows')
print(f'\nLoaded {len(train_data_by_band)} bands')

## Model reference (paper)

| Paper | This notebook |
|-------|----------------|
| **SSA:** window L, SVD, first r components, diagonal averaging | `ssa_trend(series, L, r)` |
| **SVR:** RBF kernel, C, ε, γ; window L_SVR → next value | `SVR(kernel='rbf')` on (window → 1 step); multi-step by iteration |
| Grid search over L_SSA, L_SVR, ε (min RMSE) | Single parameter set; grid search optional |
| Metrics: RMSE, MAE | MAE, RMSE, MASE (same as notebook 10) |

## SSA (Singular Spectrum Analysis)

Paper: embedding with window length L, SVD, group first r components, diagonal averaging to get trend series.

In [ ]:
def ssa_trend(series: np.ndarray, L: int, r: int = 1):
    """
    Extract trend via SSA: embed -> SVD -> reconstruct from first r components -> diagonal averaging.
    L = window length, r = number of components (small r = smoother trend).
    """
    n = len(series)
    if n < L or L < 2 or r < 1:
        return series.copy()
    K = n - L + 1
    # Trajectory matrix (L x K)
    X = np.column_stack([series[i:i+L] for i in range(K)])
    U, s, Vt = np.linalg.svd(X, full_matrices=False)
    # Reconstruct from first r components
    Xr = (U[:, :r] * s[:r]) @ Vt[:r, :]
    # Diagonal averaging -> time series
    trend = np.zeros(n)
    for k in range(n):
        i_lo, i_hi = max(0, k - K + 1), min(k, L - 1)
        if i_lo <= i_hi:
            trend[k] = np.mean([Xr[i, k - i] for i in range(i_lo, i_hi + 1)])
    return trend

## SVR trend prediction (paper)

Train SVR on (window of trend -> next value). Multi-step forecast: predict 1 step, append to window, repeat.

In [ ]:
def build_svr_xy(trend: np.ndarray, L_svr: int):
    """Build (X, y) for SVR: each row = L_svr values, target = next value."""
    X, y = [], []
    for i in range(len(trend) - L_svr):
        X.append(trend[i:i+L_svr])
        y.append(trend[i+L_svr])
    return np.array(X), np.array(y)


def predict_multistep(svr, scaler_x, scaler_y, last_window: np.ndarray, steps: int):
    """Predict next `steps` values iteratively (paper eq. 7). Window in original scale."""
    preds = []
    w = last_window.copy().astype(float)
    for _ in range(steps):
        x = scaler_x.transform(w.reshape(1, -1))
        p_scaled = svr.predict(x)[0]
        p_scaled = np.clip(p_scaled, 0, 1)
        p_orig = float(scaler_y.inverse_transform([[p_scaled]])[0, 0])
        p_orig = np.clip(p_orig, 0, 100)
        preds.append(p_orig)
        w = np.roll(w, -1)
        w[-1] = p_orig
    return np.array(preds)


def fit_svr_and_predict_per_freq(train_series: np.ndarray, test_series: np.ndarray,
                                  L_ssa: int, r_ssa: int, L_svr: int,
                                  C: float, epsilon: float, gamma: float,
                                  forecast_horizon: int, n_test_days: int):
    """SSA trend extraction + SVR fit on train trend; predict 24h per test day (iterative)."""
    full = np.concatenate([train_series, test_series])
    trend_full = ssa_trend(full, L_ssa, r_ssa)
    n_train = len(train_series)
    trend_train = trend_full[:n_train]
    trend_test = trend_full[n_train:]
    if len(trend_train) < L_svr + 1 or len(trend_test) < n_test_days * forecast_horizon:
        return None, None
    X_tr, y_tr = build_svr_xy(trend_train, L_svr)
    scaler_x = MinMaxScaler(feature_range=(0, 1))
    scaler_y = MinMaxScaler(feature_range=(0, 1))
    X_tr_s = scaler_x.fit_transform(X_tr)
    y_tr_s = scaler_y.fit_transform(y_tr.reshape(-1, 1)).ravel()
    svr = SVR(kernel='rbf', C=C, epsilon=epsilon, gamma=gamma, n_jobs=-1)
    svr.fit(X_tr_s, y_tr_s)
    preds_per_day = []
    trend_hist = list(trend_train)
    for d in range(n_test_days):
        window = np.array(trend_hist[-L_svr:])
        pred_24 = predict_multistep(svr, scaler_x, scaler_y, window, forecast_horizon)
        preds_per_day.append(pred_24)
        trend_hist.extend(pred_24)
    y_pred = np.array(preds_per_day)
    y_true = trend_test[:n_test_days * forecast_horizon].reshape(n_test_days, forecast_horizon)
    return y_pred, y_true

In [ ]:
def calculate_mae(y_true, y_pred):
    return mean_absolute_error(y_true.flatten(), y_pred.flatten())

def calculate_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true.flatten(), y_pred.flatten()))

def calculate_mase(y_true, y_pred, y_train):
    mae = np.mean(np.abs(y_true.flatten() - y_pred.flatten()))
    if len(y_train) > 1:
        scale = np.mean(np.abs(np.diff(y_train.flatten()))) or 1.0
    else:
        scale = 1.0
    return mae / scale

## SSA-SVR parameters and training

Paper: L_SSA (window SSA), L_SVR (window SVR), ε (epsilon), C and γ (gamma). We use one reasonable set; optional grid search can be added.

In [ ]:
L_SSA = 8
r_SSA = 1
L_SVR = 24
C_SVR = 3.0
epsilon_SVR = 0.05
gamma_SVR = 'scale'

n_test_days = 3
all_y_pred, all_y_true = [], []
all_y_train = []

for band in class_options:
    if band not in train_data_by_band:
        continue
    train_df = train_data_by_band[band]
    test_df = test_data_by_band[band]
    series_per_freq = get_series_per_freq(train_df, test_df)
    for freq, (tr, te) in series_per_freq.items():
        res = fit_svr_and_predict_per_freq(tr, te, L_SSA, r_SSA, L_SVR,
                                            C_SVR, epsilon_SVR, gamma_SVR,
                                            FORECAST_HORIZON, n_test_days)
        if res[0] is not None:
            y_pred, y_true = res
            all_y_pred.append(y_pred)
            all_y_true.append(y_true)
            full_tr = np.concatenate([tr, te])
            trend_tr = ssa_trend(full_tr, L_SSA, r_SSA)[:len(tr)]
            all_y_train.append(trend_tr)

if not all_y_pred:
    raise ValueError('No predictions; check data and parameters.')

y_pred_ssa_svr = np.vstack(all_y_pred)
y_test_trend = np.vstack(all_y_true)
y_train_trend = np.concatenate(all_y_train)
print(f'SSA-SVR predictions shape: {y_pred_ssa_svr.shape}')
print(f'Test trend shape: {y_test_trend.shape}')

In [ ]:
mae_ssa_svr = calculate_mae(y_test_trend, y_pred_ssa_svr)
rmse_ssa_svr = calculate_rmse(y_test_trend, y_pred_ssa_svr)
mase_ssa_svr = calculate_mase(y_test_trend, y_pred_ssa_svr, y_train_trend)

results = [{'Model': 'SSA-SVR', 'MAE': mae_ssa_svr, 'RMSE': rmse_ssa_svr, 'MASE': mase_ssa_svr}]
results_df = pd.DataFrame(results)
results_df['MAE'] = results_df['MAE'].round(4)
results_df['RMSE'] = results_df['RMSE'].round(4)
results_df['MASE'] = results_df['MASE'].round(4)
print('\n' + '='*60)
print('RESULTS (trend prediction; metrics on trend vs trend)')
print('='*60)
print(f'Lookback: {LOOKBACK}h, Forecast: {FORECAST_HORIZON}h')
print(results_df.to_string(index=False))
display(results_df)

## Predicted vs actual (dashed)

In [ ]:
hours = np.arange(FORECAST_HORIZON)
n_show = min(3, len(y_pred_ssa_svr))
fig, axes = plt.subplots(n_show, 1, figsize=(12, 4*n_show))
if n_show == 1:
    axes = [axes]
for i in range(n_show):
    ax = axes[i]
    ax.plot(hours, y_test_trend[i], 'k--', linewidth=2.5, label='Actual (trend)', alpha=0.8)
    ax.plot(hours, y_pred_ssa_svr[i], '-', color='#3498db', linewidth=1.5, label='SSA-SVR pred')
    ax.set_xlabel('Hour of day')
    ax.set_ylabel('Air time utilization (%)')
    ax.set_title(f'Test sample {i+1}: Predicted (solid) vs Actual trend (dashed)')
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hours, y_test_trend.mean(axis=0), 'k--', linewidth=2.5, label='Actual (mean trend)')
ax.plot(hours, y_pred_ssa_svr.mean(axis=0), '-', color='#3498db', linewidth=1.5, label='SSA-SVR (mean)')
ax.set_xlabel('Hour of day')
ax.set_ylabel('Air time utilization (%)')
ax.set_title('Mean 24h profile: predicted vs actual trend')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()